# # Xplore: A LangSmith Example for PDF and YouTube Content

This notebook demonstrates how to use LangSmith to load and process content from both PDF documents and YouTube videos. It includes:
- Loading PDF files with OCR fallback for image-based content
- Downloading YouTube video converted to audio and transcribing it using Faster Whisper
- Combining and chunking the extracted content for further processing

# # PDF Loading with OCR Fallback

In [ ]:
from langchain_core.messages.block_translators import groq
from sqlalchemy.orm import persistence
!pip install langchain --quiet
!pip install langchain-core --quiet
!pip install langchain-community --quiet
!pip install pypdf --quiet
!pip install pytesseract --quiet
!pip install pillow --quiet
!pip install pdf2image --quiet
!brew install tesseract --quiet 2>/dev/null || echo "Tesseract may need manual install"
!brew install poppler --quiet 2>/dev/null || echo "Poppler may need manual install"

In [ ]:
import pytesseract
import os
from PIL import Image
from pdf2image import convert_from_path
from langchain_community.document_loaders import PyPDFLoader, YoutubeLoader
from langchain_core.documents import Document


def load_pdf_with_ocr(pdf_path):
    """
    Load PDF and extract text from both text-based and image-based pages using OCR.
    Falls back to OCR if standard text extraction returns empty content.
    """
    documents = []

    # First try standard PDF loader
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
    except Exception as e:
        print(f"Standard PDF loading failed: {e}")
        documents = []

    # Check if pages have meaningful content, if not use OCR
    has_content = any(len(doc.page_content.strip()) > 100 for doc in documents)

    if not has_content:
        print("No text found in standard extraction, trying OCR...")
        try:
            # Convert PDF pages to images and extract text using OCR
            images = convert_from_path(pdf_path)
            ocr_documents = []

            for page_num, image in enumerate(images):
                # Extract text using Tesseract OCR
                text = pytesseract.image_to_string(image)

                # Create a document for this page
                doc = Document(
                    page_content=text,
                    metadata={
                        "source": pdf_path,
                        "page": page_num,
                        "extraction_method": "OCR"
                    }
                )
                ocr_documents.append(doc)

            documents = ocr_documents
            print(f"Successfully extracted {len(documents)} pages using OCR")
        except Exception as e:
            error_msg = str(e)
            print(f"OCR extraction failed: {e}")
            if "poppler" in error_msg.lower() or "page count" in error_msg.lower():
                print("\n❌ Poppler is not installed or not in PATH!")
                print("Fix this by running: brew install poppler")
                print("\nIf Tesseract is also missing, run: brew install tesseract")
            elif "tesseract" in error_msg.lower():
                print("\n❌ Tesseract is not installed!")
                print("Fix this by running: brew install tesseract")
            else:
                print("Please ensure both Tesseract and Poppler are installed on your system.")

    return documents


In [ ]:
# Usage
pdf_path = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf"
# pdf_path = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/audhil-report.pdf"
pdf_pages = load_pdf_with_ocr(pdf_path)
if pdf_pages:
    print(f"Loaded {len(pdf_pages)} pages")
    print("First page content preview:")
    print(pdf_pages[0].page_content[:1500])
    #print(pdf_pages[0])

# # Youtube video -> audio -> transcript

In [ ]:
# Fix PATH for Homebrew tools (especially important for notebook environments)
import os
import subprocess

# Ensure homebrew bin is in PATH
homebrew_paths = ["/opt/homebrew/bin", "/usr/local/bin"]
current_path = os.environ.get("PATH", "").split(":")
for path in homebrew_paths:
    if path not in current_path:
        current_path.insert(0, path)
os.environ["PATH"] = ":".join(current_path)

# Install system dependencies for YouTube audio processing
!brew install ffmpeg --quiet 2>/dev/null || echo "FFmpeg may need manual install"
# Install Python dependencies
!pip install yt_dlp --quiet
!pip install --upgrade pip
!pip install pydub --quiet
!pip install faster-whisper --quiet
!pip install torch --quiet
!pip install ffmpeg-python --quiet

In [ ]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers.audio import FasterWhisperParser  # to generate transcript from audio
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

# Set FFmpeg path explicitly for Homebrew installation
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

# Also set explicit ffmpeg location for yt-dlp
import subprocess

ffmpeg_path = None
ffprobe_path = None

try:
    ffmpeg_path = subprocess.check_output(["which", "ffmpeg"], text=True).strip()
    ffprobe_path = subprocess.check_output(["which", "ffprobe"], text=True).strip()

    # Set environment variables for the entire process
    os.environ["FFMPEG_LOCATION"] = ffmpeg_path
    os.environ["FFPROBE_LOCATION"] = ffprobe_path
    os.environ["PATH"] = f"{os.path.dirname(ffmpeg_path)}:" + os.environ.get("PATH", "")

    print(f"✓ FFmpeg found at: {ffmpeg_path}")
    print(f"✓ FFprobe found at: {ffprobe_path}")

    # Create yt-dlp config to locate ffmpeg
    yt_dlp_config_dir = os.path.expanduser("~/.config/yt-dlp")
    os.makedirs(yt_dlp_config_dir, exist_ok=True)

    yt_dlp_config = os.path.join(yt_dlp_config_dir, "config.txt")
    with open(yt_dlp_config, "w") as f:
        f.write(f"# Auto-generated config\nffmpeg-location {os.path.dirname(ffmpeg_path)}\n")
    print(f"✓ yt-dlp config created at {yt_dlp_config}")

except Exception as e:
    print(f"Error: Could not locate ffmpeg/ffprobe: {e}")
    print("Please install: brew install ffmpeg")


In [ ]:
url = "https://www.youtube.com/watch?v=uFhDGagZzjs"
save_dir = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube"

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Load YouTube audio and transcribe
try:
    print(f"Downloading and transcribing: {url}")
    loader = GenericLoader(
        YoutubeAudioLoader([url], save_dir),
        FasterWhisperParser()
    )
    yt_docs = loader.load()
    print(f"\n✓ Successfully loaded {len(yt_docs)} documents from YouTube")
    if yt_docs:
        print("\nTranscription preview:")
        print(yt_docs[0].page_content[:500])
except Exception as e:
    error_str = str(e)
    print(f"\n❌ Error loading YouTube audio: {e}")
    print(f"FFmpeg location: {ffmpeg_path}")
    print("\nTroubleshooting:")
    print("1. Ensure FFmpeg is installed: brew install ffmpeg")
    print("2. Verify PATH: which ffmpeg, which ffprobe")
    print("3. Test FFmpeg: ffmpeg -version")
    if "ffmpeg" in error_str.lower() or "ffprobe" in error_str.lower():
        print("\nThe issue is FFmpeg-related. Try restarting the notebook.")


In [ ]:
len(yt_docs)

In [ ]:
len(pdf_pages)

In [ ]:
combined_docs = pdf_pages + yt_docs
print(f"Total combined documents: {len(combined_docs)}")

In [ ]:
## Chunking

In [ ]:
!pip install langchain_text_splitters --quiet

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 1024
chunk_overlap = 200
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunked_docs = splitter.split_documents(combined_docs)



In [ ]:
chunked_docs[0]

In [ ]:
print(f"Total chunked documents: {len(chunked_docs)}")

# # Embeddings

In [ ]:
!pip install -U langchain-huggingface
!pip install sentence-transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

all_minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
multilingual_embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

In [ ]:
sentence1 = 'I like India, and it is my motherland'
sentence2 = 'I like Malaysia, it lies in east of India'
sentence3 = 'One of the best places to visit in India is the Taj Mahal'

In [ ]:
embedding1 = all_minilm_embeddings.embed_query(sentence1)
embedding2 = all_minilm_embeddings.embed_query(sentence2)
embedding3 = all_minilm_embeddings.embed_query(sentence3)

In [ ]:
print(f"Embedding 1 length: {len(embedding1)}")
print(f"Embedding 2 length: {len(embedding2)}")
print(f"Embedding 3 length: {len(embedding3)}")

In [ ]:
import numpy as np

np.dot(embedding1, embedding2)

In [ ]:
np.dot(embedding1, embedding3)

# # Vector DB

In [ ]:
# understand HNSW algorithm and how to use it with langchain

In [ ]:
!pip install chromadb --quiet
!pip install faiss-cpu --quiet
from langchain_community.vectorstores import Chroma

# persist_directory = '/db/chroma/'
vectordb = Chroma.from_documents(documents=chunked_docs, embedding=multilingual_embeddings)


In [ ]:
question = "How to do testing in Embedded Systems?"
# question = "India is my country"

In [ ]:
vectordb.similarity_search(question, k=3)  # top 3

In [ ]:
vectordb.similarity_search_with_score(question, k=3)  # top 3 with scores

# # Retrieval

In [ ]:
vectordb.max_marginal_relevance_search(question, k=3,
                                       fetch_k=10)  # k = 3 top results, fetch_k = 10 fetch top 10 and then re-rank them to get top 3 with more diversity

# # Meta data filtering

In [ ]:
vectordb.similarity_search(question, k=3, filter={
    'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf'})

# # Groq

In [ ]:
!pip install langchain-groq --quiet

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=250
)

In [ ]:
response = llm.invoke("Write about India!")
response.content

# # Prompt Engineering - Role; Instruction; Context; Examples;

In [ ]:
system_prompt = ("You are an assistant for question-answering tasks. "
                 "Use the following pieces of retrieved context to answer the question. "
                 "If you don't know the answer, just say that you don't know. "
                 "Use three sentences maximum and keep the answer concise."
                 )

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

system_message = SystemMessage(content=system_prompt)

In [ ]:
docs = vectordb.similarity_search_with_score(question, k=5)

In [ ]:
import pandas as pd

_docs = pd.DataFrame(
    [(question, doc[0].page_content, doc[0].metadata.get('source'), doc[0].metadata.get('page'), doc[1]) for doc in
     docs],
    columns=['query', 'paragraph', 'document', 'page_number', 'relevant_score']
)

_docs

In [ ]:
context = "\n\n".join(_docs['paragraph'])
context

In [ ]:
human_message = HumanMessage(content=context + question)
human_message

In [ ]:
result = llm.invoke([system_message, human_message])
result

In [ ]:
result.content

In [ ]:
question2 = "What is meant by IC?"

In [ ]:
docs2 = vectordb.similarity_search_with_score(question2, k=5)
_docs2 = pd.DataFrame(
    [(question, doc[0].page_content, doc[0].metadata.get('source'), doc[0].metadata.get('page'), doc[1]) for doc in
     docs2],
    columns=['query', 'paragraph', 'document', 'page_number', 'relevant_score']
)
context2 = "\n\n".join(_docs2['paragraph'])
human_message = HumanMessage(content=context2 + question2)
result2 = llm.invoke([system_message, human_message])
result2.content

# # LangGraph - Memory

In [ ]:
!pip install langgraph --quiet

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

workflow = StateGraph(state_schema=MessagesState)

In [ ]:
# Define the function that calls the model

def call_model(state: MessagesState):
    system_prompt = (
        "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question."
        "Answer all questions to the best of your ability."
    )
    # Prepend the system message to the current conversation history
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)
    return {"messages": response}

# define the node and edge
workflow.add_node("model", call_model)
workflow.add_edge(START, "model")

# add simple in-memory checkpointer
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


In [ ]:
app.invoke({"messages": [HumanMessage(content=context + question)]}, config={"configurable":{"thread_id":"1"}})

In [ ]:
app.invoke({"messages": [HumanMessage(content="What did I ask you?")]}, config={"configurable":{"thread_id":"1"}})